In [1]:
import torch
import torchvision.transforms as transforms
from torch2trt import TRTModule
import pycuda.autoinit
from utils.yolo import TRT_YOLO
import cv2
import PIL.Image
import numpy as np
import time

# 1. 設定道路跟隨的 PyTorch 預處理參數
device = torch.device('cuda')
mean = torch.Tensor([0.485, 0.456, 0.406]).cuda().half()
std = torch.Tensor([0.229, 0.224, 0.225]).cuda().half()

def preprocess_road(image):
    # 將攝影機的 BGR 格式轉為 RGB，並縮放為 ResNet 需要的 224x224
    image = cv2.resize(image, (224, 224))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = PIL.Image.fromarray(image)
    image = transforms.functional.to_tensor(image).to(device).half()
    image.sub_(mean[:, None, None]).div_(std[:, None, None])
    return image[None, ...]

print("開始載入模型，這可能需要幾分鐘...")

# 2. 載入 Project 5 的道路跟隨模型 (ResNet-18)
model_road = TRTModule()
model_road.load_state_dict(torch.load('best_steering_model_final_trt.pth'))
print("✅ 道路跟隨模型載入完成")

# 3. 載入 Project 6 的路牌辨識模型 (YOLOv4-tiny)
trt_yolo = TRT_YOLO("yolov4-tiny-new", (416, 416), 4)
print("✅ 路牌辨識模型載入完成")

開始載入模型，這可能需要幾分鐘...
✅ 道路跟隨模型載入完成
✅ 路牌辨識模型載入完成


In [14]:
from jetbot import Camera, bgr8_to_jpeg
from jetbot import Robot
import ipywidgets
from IPython.display import display

# 初始化設備
robot = Robot()
camera = Camera.instance(width=224, height=224, fps=10)

# 建立微調滑桿 (保留 Project 5 的參數設定)
speed_gain_slider = ipywidgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.15, description='speed gain')
steering_gain_slider = ipywidgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.09, description='steering gain')
steering_dgain_slider = ipywidgets.FloatSlider(min=0.0, max=0.5, step=0.001, value=0.01, description='steering kd')
steering_bias_slider = ipywidgets.FloatSlider(min=-0.3, max=0.3, step=0.01, value=-0.03, description='steering bias')

# 建立顯示影像的 Widget
image_widget = ipywidgets.Image(format='jpeg', width=416, height=416)

print("請微調以下參數以適應您的場地：")
display(ipywidgets.VBox([
    speed_gain_slider, 
    steering_gain_slider, 
    steering_dgain_slider, 
    steering_bias_slider,
    image_widget
]))

請微調以下參數以適應您的場地：


In [15]:
def getNearest(signs, current_time, stop_ignore_time, railway_ignore_time):
    if not len(signs):
        return [0, -1]
        
    cls_id = signs[0][1]
    
    # 根據 Project 6 的 Class ID: Road close(0), Slow(1), Railway(2), Stop(3)
    if (cls_id == 3 and (current_time > stop_ignore_time)) \
        or (cls_id == 2 and (current_time > railway_ignore_time)) \
        or (cls_id in [0, 1]): 
        return signs[0]

    return getNearest(signs[1:], current_time, stop_ignore_time, railway_ignore_time) if len(signs) >= 2 else [0, -1]

In [16]:
import time
import numpy as np
import cv2

# ==========================================
# 🚨 1. 清除背景幽靈進程
# ==========================================
try:
    camera.unobserve_all()
except:
    pass

# ==========================================
# 🚨 2. 全域變數初始化
# ==========================================
if 'stop_ignore' not in globals():
    stop_ignore = 0
if 'railway_ignore' not in globals():
    railway_ignore = 0
if 'stop_until_time' not in globals():
    stop_until_time = 0
if 'slow_until_time' not in globals():
    slow_until_time = 0
if 'angle_last' not in globals():
    angle_last = 0.0

frame_count = 0
is_processing = False

def execute(change):
    global angle_last, is_processing, frame_count
    global stop_until_time, stop_ignore, railway_ignore, slow_until_time 
    
    if is_processing:
        return
    is_processing = True

    try:
        t = time.time()
        image = change['new']
        img_draw = image.copy() 
        
        # ====================================================
        # 🛑 狀態機 1：強制煞車期間 (非阻塞式，吃掉舊畫面防卡頓)
        # ====================================================
        if t < stop_until_time:
            robot.left_motor.value = 0.0
            robot.right_motor.value = 0.0
            cv2.putText(img_draw, "STOPPING...", (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)
            image_widget.value = bgr8_to_jpeg(img_draw)
            return  

        # ====================================================
        # 任務一：道路跟隨 (🌟 恢復最純粹的 PID，不再干擾轉彎)
        # ====================================================
        xy = model_road(preprocess_road(image)).detach().float().cpu().numpy().flatten()
        x = xy[0]
        y = (0.5 - xy[1]) / 2.0
        
        point_x = int((xy[0] + 1.0) / 2.0 * 224)
        point_y = int((xy[1] + 1.0) / 2.0 * 224)
        cv2.circle(img_draw, (point_x, point_y), 10, (0, 255, 0), -1) 
        
        angle = np.arctan2(x, y)
        pid = angle * steering_gain_slider.value + (angle - angle_last) * steering_dgain_slider.value
        angle_last = angle
        
        steering = pid + steering_bias_slider.value
        base_speed = speed_gain_slider.value
        
        # 完美保留方向盤的速差
        speed_left = base_speed + steering
        speed_right = base_speed - steering

        # ====================================================
        # 任務二：路標辨識 (YOLO) 
        # ====================================================
        ALERT_WIDTH = 30
        
        if frame_count < 2:
            frame_count += 1
        else:
            frame_count = 0
            boxes, confs, clss = trt_yolo.detect(image, conf_th=0.2)
            signs = []
            
            for i, (box, cls) in enumerate(zip(boxes, clss)):
                width = box[2] - box[0]
                signs.append([width, cls])
                cv2.rectangle(img_draw, (box[0], box[1]), (box[2], box[3]), (0, 0, 255), 2)
                cv2.putText(img_draw, f"ID:{cls} W:{width}", (box[0], max(0, box[1]-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            if len(signs) > 0:
                signs.sort(reverse=True, key=lambda x : x[0])
                sign = signs[0] 
                
                # --- 判斷號誌邏輯 ---
                if sign[1] == 3 and sign[0] > ALERT_WIDTH:      
                    if t > stop_ignore:
                        print(f"🛑 看到 Stop！啟動煞車 3 秒")
                        stop_until_time = t + 3.0    
                        stop_ignore = t + 10.0       
                        return 
                    
                elif sign[1] == 2 and sign[0] > (ALERT_WIDTH - 20): 
                    if t > railway_ignore:
                        print(f"🚆 看到平交道！啟動煞車 5 秒")
                        stop_until_time = t + 5.0
                        railway_ignore = t + 10.0
                        return 
                
                elif sign[1] == 1 and sign[0] > ALERT_WIDTH:        
                    slow_until_time = t + 2.0  
                    
                elif sign[1] == 0 and sign[0] > ALERT_WIDTH:      
                    robot.stop()
                    image_widget.value = bgr8_to_jpeg(img_draw)
                    return 

        # ====================================================
        # 任務三：馬達輸出 (拔除所有最低速限制)
        # ====================================================
        if t < slow_until_time:
            # 您實測 0.15 就夠，假設 base_speed 約 0.25，這裡用 0.6 的倍率剛好會降到 0.15 左右
            speed_ratio = 0.6   
        else:
            speed_ratio = 1.0   

        final_left = speed_left * speed_ratio
        final_right = speed_right * speed_ratio

        # 直接輸出給馬達，絕不干擾 PID 的速差
        robot.left_motor.value = max(min(final_left, 1.0), 0.0)
        robot.right_motor.value = max(min(final_right, 1.0), 0.0)
        image_widget.value = bgr8_to_jpeg(img_draw)

    finally:
        is_processing = False

In [ ]:
import time
from IPython.display import clear_output

robot.set_motors(0.15, 0.15) 
time.sleep(0.1)
print("🚗 引擎啟動，進入無窮迴圈模式！")

try:
    while True:
        execute({'new': camera.value})
        time.sleep(0.01) 
except KeyboardInterrupt:
    robot.stop()
    print("🛑 已手動停止！")

🚗 引擎啟動，進入無窮迴圈模式！
🚆 看到平交道！啟動煞車 5 秒
🚆 看到平交道！啟動煞車 5 秒


In [12]:
# 解除相機綁定並停止馬達
camera.unobserve(execute, names='value')
time.sleep(0.1)
robot.stop()
print("🛑 車輛已安全停止")

# 測試完全結束後才執行這行關閉相機 (否則相機指示燈會一直亮著)
# camera.stop()

🛑 車輛已安全停止


In [13]:
camera.stop()

In [15]:
!free -m

              total        used        free      shared  buff/cache   available
Mem:           3964        3307         217           4         439         514
Swap:          6078        1048        5029
